In [1]:
import pandas as pd
import os as os
import json
import re as r
from pprint import pprint
import unicodedata as uncd
import requests
import rdflib as rdf
from rdflib.namespace import SDO, RDF, RDFS
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore
import SPARQLWrapper as sw

In [2]:
entities= dict()

with open("dev_data.json/dev_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    #data = uncd.normalize("NFKC", data)

In [ ]:
for p in data:
    narrat = p['narration']
    for z in p['entity_ref_dict']:
        ent = z
        eve = p['entity_ref_dict'][z]
        narrat = r.sub(ent, eve, narrat)
    print(narrat)

In [3]:
for x in data:
    narration_unfixed_encoding = x['narration']
    narration_fixed = uncd.normalize("NFKC", narration_unfixed_encoding)
    narration_fixed = "".join(c for c in narration_fixed if uncd.category(c)[0] != "C")

    narration = narration_fixed
    main_event = x['Event_Name']
    main_event_triples = dict()

    for y in x['keep_triples']:
        main_event_triples[y[1]] = y[2]

    main_event_triples['type'] = x['types']
    main_event_triples['appears in'] = narration
    entities[main_event] = main_event_triples

    for z in x['entity_ref_dict']:
        entity = z
        event = x['entity_ref_dict'][z]

        narration = r.sub(entity, event, narration)

    for z in x['entity_ref_dict']:
        event = x['entity_ref_dict'][z]
        entities[event] = {'appears in':narration}

    if narration not in entities.keys():
        entities[narration] = {'type':'context_narration'}

In [3]:
idx=0
for x in data:
    if x["Event_Name"] == "Gallant Bloom Handicap":
        print(idx)
    else:
        idx+=1

12


In [ ]:
pprint(entities)

In [31]:
API_URL = "https://rel.cs.ru.nl/api"
text_doc = "Brunei at the 2013 World Aquatics Championships"
# Example EL.
el_result = requests.post(API_URL, json={
    "text": text_doc,
    "spans": []
}).json()

# Example ED.
ed_result = requests.post(API_URL, json={
    "text": text_doc,
    "spans": []
}).json()

In [32]:
print (el_result)
print (ed_result)

[[0, 6, 'Brunei', 'Brunei', 0.40384330704137344, 0.9925177097320557, 'LOC'], [19, 28, 'World Aquatics Championships', 'FINA_World_Aquatics_Championships', 0.6182361875539115, 0.8689966201782227, 'MISC']]
[[0, 6, 'Brunei', 'Brunei', 0.40384330704137344, 0.9925177097320557, 'LOC'], [19, 28, 'World Aquatics Championships', 'FINA_World_Aquatics_Championships', 0.6182361875539115, 0.8689966201782227, 'MISC']]


In [36]:
base_uri = "https://github.com/ElleEsseDi/Semantic-Digital-Library-Project"

graph = rdf.Graph()

In [ ]:
dictionary_of_objects = dict()
types = set()

In [32]:
id=1
for ent in entities.keys():
    dictionary_of_objects[ent] = f"{base_uri}/{id}"
    id+=1


In [ ]:
print (dictionary_of_objects)
print(len(dictionary_of_objects))

In [ ]:
print(dictionary_of_objects['Brunei at the 2013 World Aquatics Championships'])

In [34]:
for ent in entities.keys():
    info = entities[ent]
    for i in info.keys():
        #print(i)
        if i == 'type':
            if type(info[i]) == str:
                types.add(info[i])
                #types.add(base_uri+'/'+r.sub(' ','_',info[i]))
            elif type(info[i]) == list:
                for t in info[i]:
                    types.add(t)
    for t in types:
        graph.add((rdf.URIRef(base_uri+'/'+r.sub(' ','_',t)), RDFS.label, rdf.Literal(t)))
print(types)

{'sports season', 'aircraft crash', 'Amendments to the Constitution of Ireland', 'criticism', 'United States Supreme Court decision', 'olympic delegation', 'context_narration', 'aviation incident', 'aviation accident', 'World Bowl', 'suicide', 'fire', 'political campaign', 'vehicle-ramming attack', 'award ceremony', 'Wikipedia controversy', 'theft', 'association football match', 'inauguration', 'legal case', 'exile', 'death', 'sports festival', 'election', 'referendums in Ireland', 'leadership election', 'Olympic delegation', 'presidential campaign', 'disappearance', 'FA Cup Final', 'Vuelta a España', 'rejected takeoff', 'labour movement', 'controversy', 'association football final', 'murder–suicide', 'tennis event', 'final'}


In [ ]:
for ent in entities.keys():
    info = entities[ent]
    for i in info.keys():
        pred = f"{base_uri}/{r.sub(' ','_',i)}"
        obj = info[i]
        if type(info[i]) == str:
            graph.add((rdf.URIRef(dictionary_of_objects[ent]), RDFS.label, rdf.Literal(ent)))

            if obj in types:
                graph.add((rdf.URIRef(dictionary_of_objects[ent]),rdf.URIRef(pred),rdf.URIRef(base_uri+'/'+r.sub(' ','_',obj))))

            elif obj not in dictionary_of_objects.keys():
                graph.add((rdf.URIRef(dictionary_of_objects[ent]),rdf.URIRef(pred),rdf.Literal(obj)))
                graph.add((rdf.URIRef(dictionary_of_objects[obj]), RDFS.label, rdf.Literal(obj)))

            elif obj in dictionary_of_objects.keys():
                graph.add((rdf.URIRef(dictionary_of_objects[ent]),rdf.URIRef(pred),rdf.URIRef(dictionary_of_objects[obj])))

        elif type(info[i]) == list:
            #if len(info[i]) == 1:
            #    graph.add((rdf.URIRef(dictionary_of_objects[ent]), rdf.URIRef(pred), rdf.Literal(info[i][0])))
            #else:
                for t in info[i]:
                    graph.add((rdf.URIRef(dictionary_of_objects[ent]), RDF.type, rdf.URIRef(base_uri+'/'+r.sub(' ','_',t))))



In [38]:
graph.serialize(destination="data1.ttl")

<Graph identifier=N57967f1ae7604204be40aa0f3796edf0 (<class 'rdflib.graph.Graph'>)>